In [29]:
import json
import pandas as pd
import ast
from tqdm import tqdm

In [2]:
df = pd.read_csv('bitcoin_tweet_2022.csv')
df

,date,text2,hashtags
0,2022-01-14,death cross bitcoin dump,['bitcoin']
1,2022-01-14,teaser bitcoin cryptocurrency furniture,"['bitcoin', 'cryptocurrency', 'furniture']"
2,2022-01-14,well time bitcoin mocked made fun upon careful,['bitcoin']
3,2022-01-14,bitcoin christmas ethereum shibthis opportunit...,"['bitcoin', 'christmas', 'ethereum', 'shib', '..."
4,2022-01-14,ainu token coming soon play earntelegram ainu ...,"['ainutoken', 'playtoearn', 'nfts', 'metaverse..."
...,...,...,...
839594,2022-12-24,making noise crypto bsc ethereum bitcoin alt c...,"['crypto', 'bsc', 'ethereum', 'bitcoin', 'alt']"
839595,2022-12-24,russiaukraine war reason biggest bitcoin sello...,['bitcoin']
839596,2022-12-24,glassnodealerts bitcoin btc percent supply las...,['bitcoin']
839597,2022-12-24,dont consider bitcoin worth much doesnt meet s...,"['bitcoin', 'decred', 'btc', 'dcr']"


In [3]:
df['hashtags'] = df['hashtags'].str.lower()
df['hashtags'] = df['hashtags'].apply(ast.literal_eval)

In [4]:
df_expanded = df.explode('hashtags')
df_expanded

,date,text2,hashtags
0,2022-01-14,death cross bitcoin dump,bitcoin
1,2022-01-14,teaser bitcoin cryptocurrency furniture,bitcoin
1,2022-01-14,teaser bitcoin cryptocurrency furniture,cryptocurrency
1,2022-01-14,teaser bitcoin cryptocurrency furniture,furniture
2,2022-01-14,well time bitcoin mocked made fun upon careful,bitcoin
...,...,...,...
839597,2022-12-24,dont consider bitcoin worth much doesnt meet s...,bitcoin
839597,2022-12-24,dont consider bitcoin worth much doesnt meet s...,decred
839597,2022-12-24,dont consider bitcoin worth much doesnt meet s...,btc
839597,2022-12-24,dont consider bitcoin worth much doesnt meet s...,dcr


### JSON FOR TIMELINE

In [5]:
threshold = 280
#280 -> 1020
#600 -> 525
#8500 -> 50
hashtag_counts = df_expanded['hashtags'].value_counts()
valid_hashtags = hashtag_counts[hashtag_counts >= threshold].index

df_filter = df_expanded[df_expanded['hashtags'].isin(valid_hashtags)]
with open("hashtag_counts.txt", "w") as f:
    f.write(df_filter['hashtags'].value_counts().to_string())
len(df_filter['hashtags'].unique())

1020

In [6]:
df_filter['date'] = pd.to_datetime(df_filter['date'])
df_filter

/tmp/ipykernel_96492/357684736.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filter['date'] = pd.to_datetime(df_filter['date'])


,date,text2,hashtags
0,2022-01-14,death cross bitcoin dump,bitcoin
1,2022-01-14,teaser bitcoin cryptocurrency furniture,bitcoin
1,2022-01-14,teaser bitcoin cryptocurrency furniture,cryptocurrency
2,2022-01-14,well time bitcoin mocked made fun upon careful,bitcoin
3,2022-01-14,bitcoin christmas ethereum shibthis opportunit...,bitcoin
...,...,...,...
839595,2022-12-24,russiaukraine war reason biggest bitcoin sello...,bitcoin
839596,2022-12-24,glassnodealerts bitcoin btc percent supply las...,bitcoin
839597,2022-12-24,dont consider bitcoin worth much doesnt meet s...,bitcoin
839597,2022-12-24,dont consider bitcoin worth much doesnt meet s...,btc


In [35]:
full_dates = pd.date_range(start=df_filter['date'].min(), end=df_filter['date'].max())
hashtags = df_filter['hashtags'].unique()
result = []

for tag in tqdm(hashtags, desc="Processing hashtags"):
    df_tag = df_filter[df_filter['hashtags'] == tag]
    daily_counts = df_tag.groupby('date').size()
    daily_counts = daily_counts.reindex(full_dates, fill_value=0)

    timeline = [
        {"date": date.strftime('%Y-%m-%d'), "count": int(count)}
        for date, count in daily_counts.items()
    ]

    total = int(daily_counts.sum())

    result.append({
        "hashtag": tag,
        "total": total,
        "timeline": timeline
    })

result_sorted = sorted(result, key=lambda x: x['total'], reverse=True)

with open("hashtag_timeline_1000.json", "w") as f:
    json.dump(result_sorted, f, indent=4)

Processing hashtags: 100%|██████████| 1020/1020 [03:20<00:00,  5.09it/s]


In [36]:
df_filter['date'] = pd.to_datetime(df_filter['date'])

full_dates = pd.date_range(start=df_filter['date'].min(), end=df_filter['date'].max())
hashtags = df_filter['hashtags'].unique()
result = []

for tag in tqdm(hashtags, desc="Processing hashtags"):
    df_tag = df_filter[df_filter['hashtags'] == tag].copy()
    df_tag['year_month'] = df_tag['date'].dt.to_period('M')
    monthly_counts = df_tag.groupby('year_month').size()
    all_months = pd.period_range(start=full_dates.min(), end=full_dates.max(), freq='M')
    monthly_counts = monthly_counts.reindex(all_months, fill_value=0)
    months_list = [
        {"month": month.strftime('%B'), "count": int(count)}
        for month, count in monthly_counts.items()
    ]

    total = int(monthly_counts.sum())

    result.append({
        "hashtag": tag,
        "total": total,
        "months": months_list
    })

result_sorted = sorted(result, key=lambda x: x['total'], reverse=True)

with open("hashtag_monthly_summary.json", "w") as f:
    json.dump(result_sorted, f, indent=4)

/tmp/ipykernel_96492/2704412094.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filter['date'] = pd.to_datetime(df_filter['date'])
Processing hashtags: 100%|██████████| 1020/1020 [03:20<00:00,  5.09it/s]


In [37]:

with open("hashtag_monthly_summary.json", "r") as f:
    data = json.load(f)
len(data)

1020

### JSON FOR WORD CLOUD

In [120]:
import nltk
from nltk.corpus import stopwords
from tqdm import tqdm
from tqdm.notebook import tqdm as tqdm_notebook

nltk.download('stopwords')
tqdm.pandas()

[nltk_data] Downloading package stopwords to /home/luis/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [121]:
df_filter

,date,text2,hashtags
0,2022-01-14,death cross bitcoin dump,bitcoin
1,2022-01-14,teaser bitcoin cryptocurrency furniture,bitcoin
1,2022-01-14,teaser bitcoin cryptocurrency furniture,cryptocurrency
2,2022-01-14,well time bitcoin mocked made fun upon careful,bitcoin
3,2022-01-14,bitcoin christmas ethereum shibthis opportunit...,bitcoin
...,...,...,...
839595,2022-12-24,russiaukraine war reason biggest bitcoin sello...,bitcoin
839596,2022-12-24,glassnodealerts bitcoin btc percent supply las...,bitcoin
839597,2022-12-24,dont consider bitcoin worth much doesnt meet s...,bitcoin
839597,2022-12-24,dont consider bitcoin worth much doesnt meet s...,btc


In [122]:
def remove_stopwords(text):
    if pd.isnull(text):
        return text
    stop_words = set(stopwords.words('english'))
    words = text.split()
    filtered_text = [word for word in words if word.lower() not in stop_words]
    return ' '.join(filtered_text)

df_filter['text2'] = df_filter['text2'].progress_apply(remove_stopwords)
df_filter

100%|██████████| 2495272/2495272 [02:47<00:00, 14898.97it/s]
/tmp/ipykernel_80639/1062152438.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filter['text2'] = df_filter['text2'].progress_apply(remove_stopwords)


,date,text2,hashtags
0,2022-01-14,death cross bitcoin dump,bitcoin
1,2022-01-14,teaser bitcoin cryptocurrency furniture,bitcoin
1,2022-01-14,teaser bitcoin cryptocurrency furniture,cryptocurrency
2,2022-01-14,well time bitcoin mocked made fun upon careful,bitcoin
3,2022-01-14,bitcoin christmas ethereum shibthis opportunit...,bitcoin
...,...,...,...
839595,2022-12-24,russiaukraine war reason biggest bitcoin sello...,bitcoin
839596,2022-12-24,glassnodealerts bitcoin btc percent supply las...,bitcoin
839597,2022-12-24,dont consider bitcoin worth much doesnt meet s...,bitcoin
839597,2022-12-24,dont consider bitcoin worth much doesnt meet s...,btc


In [123]:
from collections import Counter

mapped = df_filter['text2'].dropna().map(lambda text: text.lower().split())
word_counts = Counter()

for word_list in tqdm(mapped, desc="Counting word - once"):
    word_counts.update(word_list)
sorted_word_counts = word_counts.most_common()

min_count = 300
valid_words = {word for word, count in word_counts.items() if count > min_count}

def filter_low_freq_words(text):
    if pd.isnull(text):
        return text
    words = text.lower().split()
    filtered = [word for word in words if word in valid_words]
    return ' '.join(filtered)

tqdm.pandas(desc="Delete word with frec <= 300")
df_filter['text2'] = df_filter['text2'].progress_apply(filter_low_freq_words)

mapped_final = df_filter['text2'].dropna().map(lambda text: text.split())

filtered_counts = Counter()
for word_list in tqdm(mapped_final, desc="Count words filtered"):
    filtered_counts.update(word_list)

with open("filtered_word_counts.txt", "w") as f:
    for word, count in filtered_counts.most_common():
        f.write(f"{word}: {count}\n")

df_filter

Delete word with frec <= 300: 100%|██████████| 2495272/2495272 [00:07<00:00, 316761.64it/s]
/tmp/ipykernel_80639/1688891019.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filter['text2'] = df_filter['text2'].progress_apply(filter_low_freq_words)
Count words filtered: 100%|██████████| 2495272/2495272 [00:05<00:00, 461744.43it/s]


,date,text2,hashtags
0,2022-01-14,death cross bitcoin dump,bitcoin
1,2022-01-14,bitcoin cryptocurrency,bitcoin
1,2022-01-14,bitcoin cryptocurrency,cryptocurrency
2,2022-01-14,well time bitcoin made fun upon careful,bitcoin
3,2022-01-14,bitcoin christmas ethereum opportunity join bt...,bitcoin
...,...,...,...
839595,2022-12-24,russiaukraine war reason biggest bitcoin sello...,bitcoin
839596,2022-12-24,bitcoin btc percent supply last active years r...,bitcoin
839597,2022-12-24,dont consider bitcoin worth much doesnt meet s...,bitcoin
839597,2022-12-24,dont consider bitcoin worth much doesnt meet s...,btc


In [ ]:
result = []

hashtags = df_filter['hashtags'].unique()

for tag in tqdm(hashtags, desc="Generate wordclouds per hashtag"):
    df_tag = df_filter[df_filter['hashtags'] == tag]
    mapped = df_tag['text2'].dropna().map(lambda text: text.split())

    word_counts = Counter()
    for word_list in mapped:
        word_counts.update(word_list)

    wordcloud = [
        {"word": word, "count": int(count)}
        for word, count in word_counts.most_common()
    ]

    result.append({
        "hashtag": tag,
        "wordcloud": wordcloud
    })

with open("hashtag_wordclouds_1000.json", "w") as f:
    json.dump(result, f, indent=4)

Generate wordclouds per hashtag: 100%|██████████| 50/50 [00:24<00:00,  2.08it/s]
